In [ ]:
# everything connected to directorys is my own preference and can be altered 
# exact code in form of .sh is found in Selfmade_Scripts

# Nanoplot

In [ ]:
NanoPlot --fastq /path/to/input.fastq --threads ${THREADS} --outdir /path/to/NanoPlot_output_directory

**Renaming for more understandable output of MultiQC**

In [ ]:
for d in "${SAMPLE}"; do
    mv /path/to/NanoPlot_output_directory/NanoStats.txt /path/to/NanoPlot_output_directory/NanoStats_${SAMPLE}.txt
done

# MultiQC

In [ ]:
multiqc_env multiqc /path/to/NanoPlot_output_directory -o /path/to/MultiQC_output_directory

<br> 
<br> 

# Minimap2 and Samtools (mapping)

In [ ]:
minimap2 -ax map-ont -Y -t ${THREADS} -R "@RG\tID:${SAMPLE}\tSM:${SAMPLE}\tPL:ONT" /path/to/genome.mmi /path/to/Raw_Sequences.fastq > /path/to/dir/${SAMPLE}.sam

sometimes the last line was empty   
for prevention put following code before the next block: ```if [[ -z "$(tail -n 1 "/path/to/dir/${SAMPLE}.sam")" ]]; then   
sed -i '${/^$/d;}' "/path/to/dir/${SAMPLE}.sam"
    fi```

In [ ]:
samtools sort -@ ${THREADS} -o "/path/to/dir/${SAMPLE}_aligned_sorted.bam" "/path/to/dir/${SAMPLE}.sam"

In [ ]:
samtools quickcheck "/path/to/dir/${SAMPLE}_aligned_sorted.bam"

In [ ]:
samtools index -@ ${THREADS} "/path/to/dir/${SAMPLE}_aligned_sorted.bam"

In [ ]:
samtools flagstat -@ ${THREADS} "/path/to/dir/${SAMPLE}_aligned_sorted.bam" > "/path/to/dir/${SAMPLE}_flagstat.txt"

In [ ]:
MAPPED=$(grep " mapped (" "/path/to/dir/${SAMPLE}_flagstat.txt" | head -1) > "/path/to/dir/mapping_summary.txt"
echo -e "${SAMPLE}\t${MAPPED}" | tee -a "/path/to/dir/mapping_summary.txt"


<br> 
<br> 

# Samtools (extracting mapped and unmapped Reads)

In [ ]:
samtools view -b -f 4 -F 0x900 -@ ${THREADS} "/path/to/dir/${SAMPLE}_aligned_sorted.bam" -o "path/to/${SAMPLE}_unmapped.bam"

In [ ]:
samtools view -b -f 4 -F 0x900 -@ ${THREADS} "/path/to/dir/${SAMPLE}_aligned_sorted.bam" -o "path/to/${SAMPLE}_mapped.bam"

In [ ]:
READS_UNMAPPED=$(conda run -n samtools_env samtools view \
    -c \
    "path/to/${SAMPLE}_unmapped.bam")

READS_MAPPED=$(conda run -n samtools_env samtools view \
    -c \
    "path/to/${SAMPLE}_mapped.bam")

> "/path/to/unmapped_read_counts.txt"
> "/path/to/mapped_read_counts.txt"

echo -e "${SAMPLE}\t${READS_UNMAPPED}" | tee -a "/path/to/unmapped_read_counts.txt"

echo -e "${SAMPLE}\t${READS_MAPPED}" | tee -a "/path/to/mapped_read_counts.txt"

In [ ]:
# BAM TO FASTQ (UNMAPPED)
samtools fastq \
    -@ "${THREADS}" \
    -T '*' \
    "path/to/${SAMPLE}_unmapped.bam" \
| gzip > "path/to/${SAMPLE}_unmapped.fastq.gz"

zcat "path/to/${SAMPLE}_unmapped.fastq.gz" \
| sed '${/^$/d;}' \
| gzip > "path/to/${SAMPLE}_unmapped.fastq.fixed.gz"

mv "path/to/${SAMPLE}_unmapped.fastq.fixed.gz" "path/to/${SAMPLE}_unmapped.fastq.gz"

In [ ]:
# BAM TO FASTQ (MAPPED)
samtools fastq \
    -@ "${THREADS}" \
    -T '*' \
    "path/to/${SAMPLE}_mapped.bam" \
| gzip > "path/to/${SAMPLE}_mapped.fastq.gz"

zcat "path/to/${SAMPLE}_mapped.fastq.gz" \
| sed '${/^$/d;}' \
| gzip > "path/to/${SAMPLE}_mapped.fastq.fixed.gz"

mv "path/to/${SAMPLE}_mapped.fastq.fixed.gz" "path/to/${SAMPLE}_mapped.fastq.gz"

<br> 
<br> 

# Kraken2

In [ ]:
# Using one Database
# TYPE = unmapped or mapped
kraken2 --db /path/to/db --threads ${THREADS} --report "/path/to/${SAMPLE}_${TYPE}.kreport" --report-minimizer-data --gzip-compressed "/path/to/${SAMPLE}_${TYPE}.fastq.gz" \
    > "/path/to/${SAMPLE}_${TYPE}.kraken2"

In [ ]:
# Using two Databases
# TYPE = unmapped or mapped
k2 classify --db /path/to/first_db,/path/to/second_db --threads ${THREADS} --report "/path/to/${SAMPLE}_${TYPE}.kreport" "/path/to/${SAMPLE}_${TYPE}.fastq.gz" \
    > "/path/to/${SAMPLE}_${TYPE}.kraken2"